In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def normalize_text(x):
    if x is None:
        return ""

    s = str(x).strip()
    s = s.replace("\\mathrm{", "")
    s = s.replace("{", "").replace("}", "")
    s = s.replace("^", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()

def isotope_pass(answer_text):
    s = normalize_text(answer_text)
    has_mass = re.search(r"\b168\b", s) is not None
    has_element = re.search(r"\bgd\b", s) is not None
    return has_mass and has_element

def classify_failure_fp_0001(answer_text):
    s = normalize_text(answer_text)

    if len(s) == 0 or not re.search(r"\d", s):
        return "hallucination"

    if "hf" in s:
        if "168" in s:
            return "failure_to_recognize_key_aspects"
        return "misapplication_of_equation_or_model"

    if not re.search(r"\bgd\b", s):
        return "incorrect_factual_knowledge"

    if not re.search(r"\b168\b", s):
        return "calculation_error"

    return "hallucination"

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        # Best-effort metadata capture; harmless if unavailable in harness
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

# ----------------------------
# Frontier Physics Task 001
# ----------------------------
@kbench.task(
    name="FP-0001 Nuclear Shell E1 Isotope Identification",
    description="Symbolic nuclear-physics reasoning under E1 selection rules and empirical scaling."
)
def fp_0001_nuclear_shell_isotope(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A short-lived, neutron-rich radioactive isotope has low-lying neutron single-particle shell-model orbitals with the following relative energies (in units of $\hbar\omega$):
- $2g_{7/2}$: $0\hbar\omega$
- $1h_{9/2}$: $-0.25\hbar\omega$
- $2f_{5/2}$: $-0.24\hbar\omega$
- $2f_{7/2}$: $-0.27\hbar\omega$

Assume:
- these are neutron states in a single-nucleon shell-model picture
- the nucleus is well described by a harmonic-oscillator-like spacing with characteristic energy $\hbar\omega$
- E1 (electric dipole) selection rules apply:
  - parity change required
  - $\Delta \ell = \pm 1$
  - $\Delta j = 0, \pm 1$, with $0 \to 0$ forbidden

Experimentally, the highest-energy E1 $\gamma$-ray observed between any pair of these four levels has energy $E_{\gamma}^{\max} = 1.858\ \mathrm{MeV}$.

From global nuclear systematics, use the empirical relation:
$\hbar\omega \approx \frac{41}{A^{1/3}}\ \mathrm{MeV}$

The semi-empirical mass formula pairing term $\delta$ is non-zero for this nucleus.

In its neutral atomic ground state, the element’s electron configuration has $4f$ and $5d$ occupancies satisfying:
$N_{5d} = \frac{1}{7}N_{4f}$

The radioactive decay mode of this isotope is dominantly $\beta^-$ emission.

Question: What is the isotope, written in $^{A}_{Z}E$ notation?

Return JSON only in the following format:
{
  "final_answer": "<isotope in symbolic form>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        if isotope_pass(final_answer):
            passed_checks = 1
        else:
            failure_mode = classify_failure_fp_0001(final_answer)

    trace = build_trace(
        task_id="fp_0001",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0001_nuclear_shell_isotope.run(kbench.llm)


In [ ]:
results = fp_0001_nuclear_shell_isotope.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df

In [ ]:
trace_df["failure_mode"].value_counts(dropna=False)

In [ ]:
trace_df.to_csv("fp_0001_trace_log.csv", index=False)